# **Question 10: The Final Boss - Architecture & The OCI Standard**

**Focus:** **containerd, runc, shim, and the OCI**

**Scenario:**
A Junior developer asks you: *"I heard Docker is deprecated in Kubernetes and they are using something called 'containerd' now. Does that mean we have to stop using Dockerfiles and Docker images?"*

To answer this, you need to explain the "Modularization" of Docker that happened a few years ago.

**Question:**
1.  **The Stack:** When you run `docker run`, the Docker Daemon (`dockerd`) doesn't actually create the container itself. It hands the request down a chain. Can you explain the roles of **containerd** and **runc**? Which one is the "High-level" runtime and which is the "Low-level" runtime?
2.  **The "Shim":** What is the purpose of the **containerd-shim**? Why do we need a tiny process to sit between containerd and the container? (Hint: Think about what happens if you restart the Docker Daemon).
3.  **The OCI:** What is the **Open Container Initiative (OCI)**, and why does it allow us to build an image with `docker` but run it with `podman`, `containerd`, or `CRI-O` without any issues?

**Part 1: The Runtime Stack**
- When you run `docker run`, what actually creates the container?
- Explain the roles of **containerd** and **runc**
- Which is the "high-level" runtime vs. "low-level" runtime?

**Part 2: The containerd-shim**
- What is the **containerd-shim** process?
- Why does it sit between containerd and the container?
- What happens if you restart the Docker daemon?

**Part 3: The OCI Standard**
- What is the **Open Container Initiative (OCI)**?
- Why can you build with `docker` but run with `podman`, `containerd`, or `CRI-O`?
- What makes container images portable across runtimes?

---

## Answer: The Container Runtime Architecture Deep Dive

### Part 1: The Runtime Stack - From `docker run` to Running Container

#### The Complete Chain

**When you execute:**
```bash
docker run -d nginx:latest
```

**The execution flow:**
```
docker CLI
    ↓
dockerd (Docker Daemon)
    ↓
containerd (High-level runtime)
    ↓
containerd-shim (Process supervisor)
    ↓
runc (Low-level runtime)
    ↓
Container Process (PID 1 inside container)
```

---

#### Component Breakdown

**1. dockerd (Docker Daemon)**
- **Role:** API server and orchestrator
- **Responsibilities:**
  - Handles Docker API requests
  - Manages images (build, pull, push)
  - Manages volumes and networks
  - Delegates container lifecycle to containerd

**What it does NOT do:** Create containers directly

---

**2. containerd (High-Level Runtime)**
- **Role:** Container lifecycle manager
- **Classification:** **High-level runtime**
- **Responsibilities:**
  - Image pulling and storage
  - Container lifecycle (create, start, stop, delete)
  - Namespace and cgroup setup
  - Network interface creation
  - Snapshot management (overlay filesystem)

**Key Operations:**
```bash
# containerd can run standalone
ctr images pull docker.io/library/nginx:latest
ctr run docker.io/library/nginx:latest nginx1

# Used directly by Kubernetes (bypassing dockerd)
crictl pull nginx:latest
crictl run nginx:latest
```

**Why it's "high-level":**
- Understands container images and layers
- Manages storage drivers (overlay2, btrfs)
- Handles networking setup
- Provides gRPC API for orchestrators

---

**3. runc (Low-Level Runtime)**
- **Role:** OCI runtime implementation
- **Classification:** **Low-level runtime**
- **Responsibilities:**
  - Creates actual Linux namespaces (PID, NET, MNT, IPC, UTS)
  - Sets up cgroups (CPU, memory, I/O limits)
  - Executes the container process
  - **Does ONE thing:** Spawn a container from an OCI bundle

**What it operates on:**
```
OCI Bundle:
    ├── config.json       (Container spec: namespaces, cgroups, mounts)
    └── rootfs/          (Container filesystem)
        ├── bin/
        ├── etc/
        └── ...
```

**Example manual usage:**
```bash
# Create OCI bundle
mkdir /mycontainer
cd /mycontainer
mkdir rootfs

# Extract image to rootfs
docker export $(docker create nginx) | tar -C rootfs -xf -

# Generate config
runc spec

# Run container
runc run mycontainer
```

**Why it's "low-level":**
- No image management
- No network setup (assumes pre-configured)
- No storage drivers
- Pure Linux primitives (namespaces, cgroups, capabilities)
- Stateless: runs, exits

---

#### High-Level vs. Low-Level Comparison

| Aspect | containerd (High-Level) | runc (Low-Level) |
|--------|------------------------|------------------|
| **Input** | Image reference (nginx:latest) | OCI bundle (config.json + rootfs) |
| **Image Handling** | Pulls, unpacks, manages layers | No image concept |
| **Networking** | Creates veth pairs, configures | Assumes network namespace exists |
| **Storage** | Manages overlay filesystem | Mounts provided rootfs |
| **API** | gRPC, long-running daemon | CLI, one-shot execution |
| **Lifecycle** | Manages container lifecycle | Creates process, exits |
| **Kubernetes Integration** | CRI plugin (containerd) | Called by containerd |

---

#### The Handoff Process

**Detailed Flow:**
```
1. dockerd receives: docker run nginx
   ├── Parses command
   ├── Resolves image: nginx:latest
   └── Calls containerd gRPC API

2. containerd receives: CreateContainer request
   ├── Pulls image (if not cached)
   ├── Unpacks image layers to snapshots
   ├── Prepares OCI bundle:
   │   ├── config.json (from image config + runtime args)
   │   └── rootfs (from merged layers)
   ├── Spawns containerd-shim
   └── Shim calls runc

3. runc receives: OCI bundle path
   ├── Creates Linux namespaces
   ├── Sets up cgroups
   ├── Mounts rootfs
   ├── Executes entrypoint (nginx)
   └── Exits (shim takes over)

4. Container is running
   ├── PID 1: nginx (inside container)
   ├── Parent: containerd-shim (outside container)
   └── dockerd tracks via containerd API
```

---

### Part 2: The containerd-shim - The Unsung Hero

#### What is containerd-shim?

**Definition:**
A small, long-lived process that acts as the **parent process** for each container, sitting between containerd and the container's PID 1.

**Process Tree:**
```bash
systemd (PID 1)
    └── containerd (PID 500)
        └── containerd-shim (PID 1000)
            └── nginx (PID 1 inside container namespace)
```

**One shim per container:**
```bash
ps aux | grep containerd-shim

# Output:
root  1000  containerd-shim -namespace moby -id abc123 ...
root  1050  containerd-shim -namespace moby -id def456 ...
root  1100  containerd-shim -namespace moby -id ghi789 ...
```

---

#### Why Do We Need the Shim?

**Problem Without Shim:**
```
containerd (parent)
    └── nginx container (child)

If containerd restarts/crashes:
    ↓
Container becomes orphaned
    ↓
Container stops or gets killed
    ↓
Data loss, downtime
```

**Solution With Shim:**
```
containerd
    └── shim (independent process)
        └── nginx container

If containerd restarts:
    ↓
Shim keeps running
    ↓
Container keeps running
    ↓
containerd reconnects to existing shims
    ↓
No disruption ✅
```

---

#### The Shim's Responsibilities

**1. Daemonless Containers**
```bash
# Start container
docker run -d nginx

# Restart Docker daemon
systemctl restart docker

# Container still running!
docker ps  # Shows container still "Up"
```

**How:**
- Shim becomes the container's parent
- containerd can restart without affecting containers
- dockerd can restart without affecting containers

**2. STDIO Handling**
```bash
docker logs <container_id>
docker attach <container_id>
```

**Shim captures:**
- Container's stdout → Forwarded to logging driver
- Container's stderr → Forwarded to logging driver
- stdin (for interactive containers) → Forwarded to container

**3. Exit Status Reporting**
```bash
docker ps -a

# Shows:
CONTAINER ID   STATUS
abc123         Exited (137) 2 minutes ago
```

**Shim:**
- Waits for container process (using waitpid)
- Captures exit code
- Reports back to containerd
- Cleans up namespaces and cgroups

**4. Resource Cleanup**
When container exits:
- Shim ensures namespaces are destroyed
- Removes temporary mounts
- Cleans up network interfaces
- Notifies containerd of termination

---

#### The Restart Scenario - What Actually Happens

**Scenario: Restart Docker Daemon**

**Before Restart:**
```bash
docker ps
# CONTAINER ID   STATUS
# abc123         Up 2 hours
```

**Process Tree:**
```
dockerd (PID 100)
    └── containerd (PID 200)
        └── shim (PID 300)
            └── nginx (PID 400 in container)
```

**During Restart:**
```bash
systemctl restart docker
# dockerd stops (PID 100 dies)
# containerd may restart
```

**Process Tree During Restart:**
```
systemd (PID 1)
    └── shim (PID 300) ← Adopted by init, still running!
        └── nginx (PID 400) ← Still running!
```

**After Restart:**
```
dockerd (PID 150, new process)
    └── containerd (PID 250, new process)
        ├── Reconnects to existing shim (PID 300)
        └── shim (PID 300) ← Same old shim!
            └── nginx (PID 400) ← Never interrupted!
```

**Result:**
```bash
docker ps
# CONTAINER ID   STATUS
# abc123         Up 2 hours  ← Uptime preserved!
```

**Key Insight:**
> The shim enables **daemonless container execution**. Containers outlive the management layer.

---

### Part 3: The OCI Standard - Universal Container Compatibility

#### What is the OCI?

**Open Container Initiative (OCI):**
- **Founded:** 2015 by Docker, CoreOS, and others
- **Goal:** Create open industry standards for container formats and runtimes
- **Governed by:** Linux Foundation

**Two Main Specifications:**

**1. OCI Runtime Specification (runtime-spec)**
- Defines how to run a container
- Specifies the `config.json` format
- Defines lifecycle operations (create, start, kill, delete)
- Reference implementation: **runc**

**2. OCI Image Specification (image-spec)**
- Defines container image format
- Layer structure (tar.gz blobs)
- Image manifest format
- Image configuration (JSON)

---

#### Why OCI Enables Universal Compatibility

**The Problem Before OCI:**
```
Docker images → Only Docker runtime
rkt images → Only rkt runtime
LXD images → Only LXD runtime

Fragmentation, vendor lock-in
```

**After OCI:**
```
OCI-compliant image → Any OCI-compliant runtime

Build with: docker, buildah, kaniko
Run with: runc, crun, kata, gVisor, containerd, CRI-O
```

---

#### OCI Image Format - The Universal Package

**Structure:**
```json
{
  "schemaVersion": 2,
  "config": {
    "digest": "sha256:abc123...",
    "mediaType": "application/vnd.oci.image.config.v1+json"
  },
  "layers": [
    {
      "digest": "sha256:layer1...",
      "mediaType": "application/vnd.oci.image.layer.v1.tar+gzip"
    },
    {
      "digest": "sha256:layer2...",
      "mediaType": "application/vnd.oci.image.layer.v1.tar+gzip"
    }
  ]
}
```

**What's Standardized:**
- Layer compression (gzip, zstd)
- Filesystem structure
- Metadata format (environment variables, entrypoint, etc.)
- Content-addressable storage (SHA256 digests)

---

#### Cross-Runtime Compatibility - Real Examples

**Build with Docker, Run with Podman:**
```bash
# Build with Docker
docker build -t myapp:v1 .
docker push registry.io/myapp:v1

# Run with Podman (on different machine)
podman pull registry.io/myapp:v1
podman run registry.io/myapp:v1

# Works seamlessly! Same OCI image format
```

**Build with Buildah, Run in Kubernetes (containerd):**
```bash
# Build with Buildah (daemonless)
buildah bud -t myapp:v1 .
buildah push myapp:v1 registry.io/myapp:v1

# Kubernetes pulls via containerd
kubectl run myapp --image=registry.io/myapp:v1

# containerd unpacks, runc runs
# No Docker involved at all
```

**Why This Works:**
1. **Same image format:** OCI-compliant layers and manifest
2. **Same runtime spec:** OCI bundle format (config.json + rootfs)
3. **Portable:** SHA256 content addressing ensures integrity
4. **Interoperable:** All runtimes understand the same standards

---

#### Answering the Junior Developer

**Question:**
> "Does Kubernetes using containerd mean we stop using Dockerfiles?"

**Answer:**
```
NO! Here's why:

1. Dockerfiles → Standard image building format
   ✅ Still used everywhere
   ✅ Supported by: docker, buildah, kaniko, buildkit

2. Docker images → OCI-compliant images
   ✅ Run on containerd (Kubernetes)
   ✅ Run on CRI-O (Kubernetes)
   ✅ Run on podman

3. What changed in Kubernetes:
   BEFORE: kubelet → dockerd → containerd → runc
   AFTER:  kubelet → containerd → runc
   
   Removed: dockerd (unnecessary middle layer)
   Kept: containerd, runc (the actual runtime)

4. Your workflow:
   Build: docker build -t app:v1 .
   Push:  docker push registry.io/app:v1
   K8s:   Uses containerd to pull and run
   
   Nothing changes for developers!
```

---

## Architecture Summary

| Component | Role | Type | Lifecycle |
|-----------|------|------|-----------|
| **dockerd** | API server, orchestrator | Daemon | Long-running |
| **containerd** | Container lifecycle manager | High-level runtime | Long-running |
| **containerd-shim** | Container supervisor | Process manager | Per-container |
| **runc** | OCI runtime | Low-level runtime | One-shot execution |

---

## The OCI Value Proposition

| Aspect | Why OCI Matters | What Problem It Solves |
|--------|----------------|------------------------|
| **Image Format** | Universal packaging | Build once, run anywhere (any OCI runtime) |
| **Runtime Spec** | Standard execution | Multiple runtime implementations (runc, crun, kata) |
| **Vendor Neutrality** | Open standard | Prevents vendor lock-in |
| **Innovation** | Extensible spec | New runtimes (gVisor, Firecracker) without breaking compatibility |

---

## Production Insights

### 1. **Why Kubernetes Removed dockerd**
```
Old: kubelet → dockerd → containerd → runc
     ↑ Extra layer, added latency, complexity

New: kubelet → containerd (CRI plugin) → runc
     ↑ Direct integration, faster, simpler
```

**Benefits:**
- ✅ Reduced latency (one less hop)
- ✅ Smaller attack surface
- ✅ containerd is lightweight (vs. full Docker daemon)
- ✅ Better resource utilization

### 2. **OCI Runtime Alternatives**

**runc (default):**
- Standard Linux containers
- Mature, stable

**crun (Red Hat):**
- Written in C (vs. runc in Go)
- Lower memory footprint
- Faster startup

**Kata Containers:**
- VM-based isolation
- Stronger security boundaries
- OCI-compliant interface

**gVisor:**
- User-space kernel
- Application sandboxing
- OCI-compliant

**All run the same OCI images!**

### 3. **The Shim in Production**

**Memory overhead:**
```bash
# Each shim: ~10MB RAM
# 100 containers = ~1GB RAM for shims alone

# Acceptable trade-off for:
# - Daemon restarts without downtime
# - Container isolation
# - Robust STDIO handling
```

---

## Interview Red Flags to Avoid

❌ "Docker is dead/deprecated"
✅ "dockerd was removed from Kubernetes, but Docker images and Dockerfiles are still the standard. containerd (which was always part of Docker) is now used directly."

❌ "containerd replaced Docker"
✅ "containerd was always inside Docker. Kubernetes just removed the dockerd layer and talks to containerd directly via CRI."

❌ "OCI images are different from Docker images"
✅ "Docker images ARE OCI images. Docker donated the spec to OCI. They're the same format."

❌ "You need Docker to build images"
✅ "Docker is one tool. Others include buildah, kaniko, buildkit. All produce OCI-compliant images."

---

## Quick Reference Card

```bash
# View the stack
pstree -p $(pidof dockerd)
# dockerd─containerd─┬─containerd-shim───nginx
#                    └─containerd-shim───redis

# Check OCI compliance
docker inspect nginx:latest --format='{{.Config.Image}}'

# Use containerd directly
ctr images pull docker.io/library/nginx:latest
ctr run --rm docker.io/library/nginx:latest test

# Manual runc execution
runc spec
runc run mycontainer

# Check shims
ps aux | grep containerd-shim
```

---

## Visual Summary

```
┌─────────────────────────────────────────┐
│         User Command: docker run        │
└─────────────────┬───────────────────────┘
                  ↓
┌─────────────────────────────────────────┐
│  dockerd (Docker Daemon)                │
│  - API server                           │
│  - Image management                     │
│  - Volume/Network management            │
└─────────────────┬───────────────────────┘
                  ↓ gRPC API
┌─────────────────────────────────────────┐
│  containerd (High-Level Runtime)        │
│  - Image pull/unpack                    │
│  - Container lifecycle                  │
│  - Snapshot management                  │
└─────────────────┬───────────────────────┘
                  ↓ Spawns
┌─────────────────────────────────────────┐
│  containerd-shim (Per Container)        │
│  - Daemonless execution                 │
│  - STDIO handling                       │
│  - Exit status reporting                │
└─────────────────┬───────────────────────┘
                  ↓ Calls
┌─────────────────────────────────────────┐
│  runc (Low-Level Runtime)               │
│  - Creates namespaces                   │
│  - Sets up cgroups                      │
│  - Spawns PID 1                         │
└─────────────────┬───────────────────────┘
                  ↓
┌─────────────────────────────────────────┐
│  Container Process (nginx, python, etc) │
└─────────────────────────────────────────┘

        All compliant with OCI Standards
```